# Assignment 5 — Spark Questions

**Objective:** Understand Spark fundamentals and perform data cleaning, transformation, and aggregation using DataFrames.

**Dataset:** `Sample - Superstore.csv` — 9,994 retail order records (Sales, Profit, Region, Category, Customer info, dates, etc.)

This notebook follows the 10 steps from the assignment brief: Spark basics → load → clean → filter → transform → aggregate → groupBy → wide transformations/shuffle (concept) → full pipeline → insights.

## Step 1: MapReduce vs Spark (Concept)

- **MapReduce** processes data in stages that read/write to **disk** between each Map and Reduce step. This makes it reliable for huge batch jobs but **slow**, especially for iterative algorithms that reuse the same data repeatedly.
- **Spark** keeps data **in memory (RAM)** across operations using Resilient Distributed Datasets (RDDs) / DataFrames, avoiding repeated disk I/O. This makes it dramatically faster (often 10–100x) for iterative workloads, interactive queries, and multi-step pipelines like the one in this assignment.
- Spark also offers a much friendlier high-level API (DataFrames, SQL) compared to writing raw Map/Reduce functions.

## Step 2: Start Spark — Create a Spark Session

A `SparkSession` is the entry point to all Spark functionality. We run in `local[*]` mode, which uses all available CPU cores on this machine — no cluster required.

In [ ]:
!pip install pyspark

In [69]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum as _sum, avg, min as _min, max as _max,
    when, coalesce, try_to_date
)

spark = SparkSession.builder \
    .appName("SuperstoreSparkAssignment") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 4.1.2


## Step 3: Load Data

We load the Superstore CSV into a Spark DataFrame.

**Note on `escape='"'` and `multiLine=True`:** while building this notebook, a plain `spark.read.csv(..., inferSchema=True)` produced the **wrong schema** — `Sales`, `Quantity`, and `Discount` were inferred as `string` instead of numeric types. Investigating showed the cause: roughly 300 rows in `Product Name` contain unescaped embedded double-quotes (e.g. a product named `5 1/2" X 4"`), which confuses Spark's default CSV parser and shifts values into the wrong columns. Adding `escape='"'` (tell Spark how quotes are escaped) and `multiLine=True` fixes the parsing and gives the correct schema. This is itself a good example of the "inconsistent/schema issues" step (Step 9 below) showing up in practice.

In [70]:
df = spark.read.csv(
    "../data/dataset.csv",
    header=True,
    inferSchema=True,
    escape='"',
    multiLine=True
)

print("Row count:", df.count())
print("Columns:", df.columns)

Row count: 9994
Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [71]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [72]:
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|11-08-2016|11-11-2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

## Step 4: Data Cleaning

- Remove duplicate rows
- Check for null values across all columns
- (This dataset turns out to have **no nulls and no exact duplicate rows** after the parsing fix above — but we still run the checks explicitly, since verifying this is part of the cleaning process, not something to assume.)

In [73]:
before_count = df.count()
df_dedup = df.dropDuplicates()
after_count = df_dedup.count()

print(f"Rows before dedup: {before_count}")
print(f"Rows after dedup:  {after_count}")
print(f"Duplicate rows removed: {before_count - after_count}")

Rows before dedup: 9994
Rows after dedup:  9994
Duplicate rows removed: 0


In [74]:
null_counts = df_dedup.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df_dedup.columns]
)
null_counts.show(vertical=True)

-RECORD 0------------
 Row ID        | 0   
 Order ID      | 0   
 Order Date    | 0   
 Ship Date     | 0   
 Ship Mode     | 0   
 Customer ID   | 0   
 Customer Name | 0   
 Segment       | 0   
 Country       | 0   
 City          | 0   
 State         | 0   
 Postal Code   | 0   
 Region        | 0   
 Product ID    | 0   
 Category      | 0   
 Sub-Category  | 0   
 Product Name  | 0   
 Sales         | 0   
 Quantity      | 0   
 Discount      | 0   
 Profit        | 0   



## Step 6: Transform Data — Rename Columns & Fix Schema

We rename columns to remove spaces/hyphens (easier to reference programmatically) and parse the `Order Date` / `Ship Date` text columns into real `date` types.

The dates in this file use **two different formats** (another real-world inconsistency): some rows are `M/d/yyyy` (e.g. `8/27/2014`) and others are `dd-MM-yyyy` (e.g. `09-01-2014`). We use `try_to_date` with `coalesce` to try the first format, and fall back to the second if the first doesn't match, without crashing on parse errors.

In [75]:
df_clean = df_dedup \
    .withColumnRenamed("Sub-Category", "SubCategory") \
    .withColumnRenamed("Postal Code", "PostalCode") \
    .withColumnRenamed("Row ID", "RowID") \
    .withColumnRenamed("Order ID", "OrderID") \
    .withColumnRenamed("Customer ID", "CustomerID") \
    .withColumnRenamed("Customer Name", "CustomerName") \
    .withColumnRenamed("Product ID", "ProductID") \
    .withColumnRenamed("Product Name", "ProductName")

df_clean = df_clean.withColumn(
    "OrderDateParsed",
    coalesce(
        try_to_date(col("Order Date"), "M/d/yyyy"),
        try_to_date(col("Order Date"), "dd-MM-yyyy")
    )
).withColumn(
    "ShipDateParsed",
    coalesce(
        try_to_date(col("Ship Date"), "M/d/yyyy"),
        try_to_date(col("Ship Date"), "dd-MM-yyyy")
    )
)

unparsed = df_clean.filter(col("OrderDateParsed").isNull()).count()
print("Order dates that failed to parse:", unparsed)

df_clean.select("Order Date", "OrderDateParsed", "Ship Date", "ShipDateParsed").show(5)

Order dates that failed to parse: 0
+----------+---------------+----------+--------------+
|Order Date|OrderDateParsed| Ship Date|ShipDateParsed|
+----------+---------------+----------+--------------+
| 8/27/2014|     2014-08-27|09-01-2014|    2014-01-09|
| 9/17/2015|     2015-09-17| 9/21/2015|    2015-09-21|
| 4/25/2016|     2016-04-25| 4/29/2016|    2016-04-29|
| 8/21/2017|     2017-08-21| 8/23/2017|    2017-08-23|
| 6/26/2015|     2015-06-26| 6/30/2015|    2015-06-30|
+----------+---------------+----------+--------------+
only showing top 5 rows


In [76]:
df_clean.printSchema()

root
 |-- RowID: integer (nullable = true)
 |-- OrderID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- PostalCode: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- OrderDateParsed: date (nullable = true)
 |-- ShipDateParsed: date (nullable = true)



## Step 5: Filter Data

Applying simple conditions on the cleaned DataFrame: by category+region, and by a sales threshold (a stand-in for "age range" filtering from the brief, since this retail dataset has no age column — Region/Category/Sales are the equivalent dimensions here).

In [77]:
print("Orders from the West region, Furniture category:")
df_clean.filter((col("Region") == "West") & (col("Category") == "Furniture")) \
    .select("OrderID", "City", "SubCategory", "Sales") \
    .show(5)

Orders from the West region, Furniture category:
+--------------+-------------+-----------+-------+
|       OrderID|         City|SubCategory|  Sales|
+--------------+-------------+-----------+-------+
|CA-2017-115154|      Seattle|     Tables| 892.98|
|CA-2014-116932|San Francisco|     Tables|272.848|
|CA-2016-153101|    Santa Ana|     Tables| 146.04|
|US-2015-165743|       Aurora|  Bookcases|145.764|
|CA-2016-129630|San Francisco|Furnishings|  24.27|
+--------------+-------------+-----------+-------+
only showing top 5 rows


In [78]:
print("High-value orders: Sales > 500")
df_clean.filter(col("Sales") > 500) \
    .select("OrderID", "Category", "Sales") \
    .orderBy(col("Sales").desc()) \
    .show(5)

High-value orders: Sales > 500
+--------------+----------+---------+
|       OrderID|  Category|    Sales|
+--------------+----------+---------+
|CA-2014-145317|Technology| 22638.48|
|CA-2016-118689|Technology| 17499.95|
|CA-2017-140151|Technology| 13999.96|
|CA-2017-127180|Technology|11199.968|
|CA-2017-166709|Technology| 10499.97|
+--------------+----------+---------+
only showing top 5 rows


## Step 7: Aggregation

Basic summary statistics across the whole cleaned dataset.

In [79]:
df_clean.select(
    count("*").alias("TotalRows"),
    _sum("Sales").alias("TotalSales"),
    avg("Sales").alias("AvgSales"),
    _min("Sales").alias("MinSales"),
    _max("Sales").alias("MaxSales"),
    avg("Profit").alias("AvgProfit")
).show()

+---------+------------------+------------------+--------+--------+-----------------+
|TotalRows|        TotalSales|          AvgSales|MinSales|MaxSales|        AvgProfit|
+---------+------------------+------------------+--------+--------+-----------------+
|     9994|2297200.8602999886|229.85800083049716|   0.444|22638.48|28.65689630778471|
+---------+------------------+------------------+--------+--------+-----------------+



## Step 8: Group Data — `groupBy()`

We group by **Region** and by **Category**, and apply a condition on the aggregated result (Spark's equivalent of SQL's `HAVING`) to keep only high-revenue categories.

In [80]:
print("Total sales & average profit by Region:")
df_clean.groupBy("Region").agg(
    _sum("Sales").alias("TotalSales"),
    avg("Profit").alias("AvgProfit"),
    count("*").alias("OrderCount")
).orderBy(col("TotalSales").desc()).show()

Total sales & average profit by Region:
+-------+------------------+------------------+----------+
| Region|        TotalSales|         AvgProfit|OrderCount|
+-------+------------------+------------------+----------+
|   West| 725457.8245000013| 33.84903181392445|      3203|
|   East| 678781.2400000009| 32.13580758426965|      2848|
|Central|501239.89079999994|17.092708781747735|      2323|
|  South|        391721.905|28.857673024691344|      1620|
+-------+------------------+------------------+----------+



In [81]:
print("Categories with total sales over $200,000 (condition on aggregated result):")
df_clean.groupBy("Category").agg(
    _sum("Sales").alias("TotalSales")
).filter(col("TotalSales") > 200000).show()

Categories with total sales over $200,000 (condition on aggregated result):
+---------------+-----------------+
|       Category|       TotalSales|
+---------------+-----------------+
|Office Supplies|719047.0320000022|
|      Furniture|741999.7953000008|
|     Technology|836154.0329999998|
+---------------+-----------------+



## Step 9: Wide Transformations & Shuffle (Concept)

- A **narrow transformation** (e.g. `filter`, `select`, `withColumn`) only needs data already present in the same partition — no data movement between partitions.
- A **wide transformation** (e.g. `groupBy`, `join`, `orderBy`, `distinct`/`dropDuplicates`) requires data with the same key to be brought together, which usually means data must move **across** partitions — this movement is called a **shuffle**.
- Shuffles are expensive: they involve disk I/O, network transfer, and serialization, so they are the main thing to watch out for when tuning Spark job performance.
- In this notebook, **`dropDuplicates()`** (Step 4) and every **`groupBy()`** (Step 8) triggered a shuffle under the hood — Spark needed to redistribute rows so matching keys/duplicate rows land in the same partition before it could compute the result.

## Step 10: Full Pipeline

Putting every step together into one continuous pipeline: **load → clean (dedup + null check) → transform (rename + cast dates) → filter → aggregate**. This mirrors what a production Spark job would look like, just expressed as one chained sequence here for clarity.

In [82]:
def run_pipeline(path):
    # Load
    raw = spark.read.csv(path, header=True, inferSchema=True, escape='"', multiLine=True)

    # Clean
    deduped = raw.dropDuplicates()

    # Transform
    transformed = deduped \
        .withColumnRenamed("Sub-Category", "SubCategory") \
        .withColumn(
            "OrderDateParsed",
            coalesce(try_to_date(col("Order Date"), "M/d/yyyy"),
                     try_to_date(col("Order Date"), "dd-MM-yyyy"))
        )

    # Filter: keep only profitable orders
    filtered = transformed.filter(col("Profit") > 0)

    # Aggregate: total sales & profit per category
    result = filtered.groupBy("Category").agg(
        _sum("Sales").alias("TotalSales"),
        _sum("Profit").alias("TotalProfit"),
        count("*").alias("ProfitableOrders")
    ).orderBy(col("TotalProfit").desc())

    return result

pipeline_result = run_pipeline("../data/dataset.csv")
pipeline_result.show()

+---------------+-----------------+------------------+----------------+
|       Category|       TotalSales|       TotalProfit|ProfitableOrders|
+---------------+-----------------+------------------+----------------+
|     Technology|716701.1479999999|184034.86629999985|            1573|
|Office Supplies|618988.7490000024|179106.05930000072|            5111|
|      Furniture|465116.9615000013| 79387.38180000002|            1374|
+---------------+-----------------+------------------+----------------+



In [ ]:
pipeline_result.toPandas().to_csv("../output/results.csv", index=False)
print("Saved to ../output/results.csv")

Saved to ../output/results.csv


## Insights & Observations

- **Data quality issue found:** ~300 rows had unescaped double-quotes inside `Product Name` (e.g. measurements like `5 1/2" X 4"`), which broke naive CSV parsing and silently corrupted the inferred schema for `Sales`, `Quantity`, and `Discount` (all became `string` instead of numeric). Adding `escape='"'` and `multiLine=True` to the CSV reader fixed this — a good reminder to always inspect `printSchema()` after loading rather than assuming `inferSchema` got it right.
- **Date inconsistency:** the `Order Date` / `Ship Date` columns mix two date formats (`M/d/yyyy` and `dd-MM-yyyy`) in the same column. Using `try_to_date` with `coalesce` parses both formats safely without crashing on the format that doesn't match.
- **No nulls or duplicate rows** were found in this dataset once parsing was fixed — so the cleaning step here was mostly about catching *schema* corruption rather than missing values.
- **Regional performance:** the **West** region has both the highest total sales and one of the higher average profits, while **Central** has noticeably lower average profit despite a respectable order count — suggesting discounting or cost issues specific to that region worth investigating further.
- **Category performance:** all three categories (Furniture, Office Supplies, Technology) individually exceed $200,000 in total sales, with **Technology** generating the most total profit among profitable orders in the pipeline run.
- **Wide vs narrow transformations:** every `groupBy` and the `dropDuplicates` call triggered a shuffle, which is the main performance cost in this kind of pipeline; on a single local machine it's invisible, but at cluster scale this is exactly where tuning (partitioning, broadcast joins, etc.) matters most.

In [84]:
spark.stop()